# Citiz — analyse de l'utilisation des véhicules
Lit la table `citiz_releves` (alimentée par `citiz_collecte`).

Le flux n'expose pas les trajets. Les indicateurs sont des **proxys** :
- **occupation** : part des relevés où le véhicule est réservé (`is_reserved`) — peut inclure des réservations à venir ;
- **début de réservation** : `is_reserved` passe de faux à vrai entre deux relevés ;
- **déplacement** : le véhicule change de `station_id` ;
- **baisse de carburant** entre deux relevés.

À interpréter après 1 à 2 jours de collecte au minimum.

In [ ]:
from pyspark.sql import functions as F, Window as W

TABLE = "citiz_releves"
raw = spark.table(TABLE)
print("Relevés :", raw.select("releve_utc").distinct().count(),
      "| du", raw.agg(F.min("releve_utc")).first()[0], "au", raw.agg(F.max("releve_utc")).first()[0])

In [ ]:
w = W.partitionBy("jeu", "vehicle_id").orderBy("releve_utc")
df = (raw
      .withColumn("resa_prec", F.lag("is_reserved").over(w))
      .withColumn("station_prec", F.lag("station_id").over(w))
      .withColumn("fuel_prec", F.lag("fuel_pct").over(w))
      .withColumn("debut_resa", (F.col("resa_prec") == False) & F.col("is_reserved"))  # noqa: E712
      .withColumn("deplace", F.col("station_prec").isNotNull() & (F.col("station_id") != F.col("station_prec")))
      .withColumn("baisse_fuel", (F.col("fuel_pct") - F.col("fuel_prec")) < -0.02)
      .withColumn("local", F.from_utc_timestamp("releve_utc", "Europe/Paris"))
      .withColumn("heure", F.hour("local"))
      .withColumn("jour_semaine", F.date_format("local", "E"))
      .withColumn("jour", F.to_date("local"))
      .cache())
df.createOrReplaceTempView("citiz_enrichi")

## Par jeu de données (territoire)

In [ ]:
par_jeu = (df.groupBy("jeu")
           .agg(F.countDistinct("vehicle_id").alias("vehicules"),
                F.round(100 * F.avg(F.col("is_reserved").cast("int")), 1).alias("occupation_pct"),
                F.sum(F.col("debut_resa").cast("int")).alias("debuts_resa"),
                F.sum(F.col("deplace").cast("int")).alias("deplacements"))
           .withColumn("resa_par_vehicule_par_jour", F.round(F.col("debuts_resa") / F.col("vehicules"), 2))
           .orderBy(F.desc("occupation_pct")))
display(par_jeu)

## Profil horaire (heure de Paris)

In [ ]:
par_heure = (df.groupBy("heure")
             .agg(F.round(100 * F.avg(F.col("is_reserved").cast("int")), 1).alias("occupation_pct"),
                  F.sum(F.col("debut_resa").cast("int")).alias("debuts_resa"))
             .orderBy("heure"))
display(par_heure)   # utiliser la vue graphique (courbe) du résultat

## Jour de la semaine × heure

In [ ]:
display(df.groupBy("jour_semaine", "heure")
          .agg(F.round(100 * F.avg(F.col("is_reserved").cast("int")), 1).alias("occupation_pct"))
          .orderBy("jour_semaine", "heure"))

## Évolution quotidienne

In [ ]:
display(df.groupBy("jour")
          .agg(F.round(100 * F.avg(F.col("is_reserved").cast("int")), 1).alias("occupation_pct"),
               F.sum(F.col("debut_resa").cast("int")).alias("debuts_resa"))
          .orderBy("jour"))

## Stations les plus / moins utilisées

In [ ]:
par_station = (df.groupBy("jeu", "station_id")
               .agg(F.countDistinct("vehicle_id").alias("vehicules"),
                    F.round(100 * F.avg(F.col("is_reserved").cast("int")), 1).alias("occupation_pct"),
                    F.sum(F.col("debut_resa").cast("int")).alias("debuts_resa"))
               .filter("vehicules >= 1"))
print("Top 15 :");    display(par_station.orderBy(F.desc("occupation_pct")).limit(15))
print("Bottom 15 :"); display(par_station.orderBy("occupation_pct").limit(15))

## Véhicules sous-utilisés (jamais réservés sur la période)

In [ ]:
par_vehicule = (df.groupBy("jeu", "vehicle_id")
                .agg(F.round(100 * F.avg(F.col("is_reserved").cast("int")), 1).alias("occupation_pct"),
                     F.sum(F.col("debut_resa").cast("int")).alias("debuts_resa"),
                     F.round(F.avg("fuel_pct"), 2).alias("carburant_moyen")))
n_rel = raw.select("releve_utc").distinct().count()
if n_rel >= 288:   # au moins ~1 jour de relevés
    display(par_vehicule.filter("debuts_resa = 0 AND occupation_pct = 0").orderBy("jeu"))
else:
    print(f"Seulement {n_rel} relevés : attendre ~1 jour (288 relevés) avant de conclure.")

## Sauvegarde des indicateurs
Écrit les tables d'agrégats pour Power BI (modèle sémantique sur le lakehouse).

In [ ]:
par_jeu.write.mode("overwrite").format("delta").saveAsTable("citiz_kpi_jeu")
par_heure.write.mode("overwrite").format("delta").saveAsTable("citiz_kpi_heure")
par_station.write.mode("overwrite").format("delta").saveAsTable("citiz_kpi_station")
par_vehicule.write.mode("overwrite").format("delta").saveAsTable("citiz_kpi_vehicule")